> **Plug LakeLogic into whatever you already run. dbt, dlt, streams, cloud, databases.**

# Integrations — LakeLogic Plays Nicely With Your Stack

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/06_integrations.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/06_integrations.ipynb)

You're not greenfield. You have a dbt project, a dlt pipeline, a Kafka topic, three Postgres replicas, and data in S3, ADLS, and GCS. The last thing you need is *another* framework that demands you rewrite everything.

This notebook shows how LakeLogic **adopts what you already have** — reuses dbt `schema.yml`, drives dlt pipelines, validates live streams, runs SQL pushdown against databases, handles incremental CDC, and pulls from every major cloud blob store with the same contract abstraction.

In [ ]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb] sseclient-py

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## 1. dbt Adapter — Reuse Your Schema Definitions

**The Problem:** You already have 200 models defined in dbt `schema.yml`. Rewriting them as LakeLogic contracts doubles your maintenance burden.

**The Solution:** `DataProcessor.from_dbt()` reads your dbt schema and creates a LakeLogic contract from it. Zero duplication.

In [ ]:
from pathlib import Path

# Write a realistic dbt schema.yml
dbt_schema = """
version: 2
models:
  - name: customers
    description: Customer master table
    columns:
      - name: customer_id
        description: Primary key
        tests:
          - not_null
          - unique
      - name: email
        description: Customer email address
        tests:
          - not_null
      - name: first_name
        description: First name
      - name: last_name
        description: Last name
      - name: country
        description: ISO country code
        tests:
          - accepted_values:
              values: ['US', 'GB', 'DE', 'FR', 'JP']
      - name: created_at
        description: Account creation timestamp
        tests:
          - not_null
"""
Path("dbt_schema.yml").write_text(dbt_schema)

# Create a LakeLogic processor from dbt definitions
proc = ll.DataProcessor.from_dbt("dbt_schema.yml", model="customers")
print("Contract created from dbt schema:")
print(f"  Dataset: {proc.contract.dataset}")
print(f"  Fields:  {[f.name for f in proc.contract.model.fields]}")
print(f"  Rules:   {len(proc.contract.quality.row_rules)} row rules")

In [ ]:
# The Proof — generate data and run through the dbt-derived contract
gen = ll.DataGenerator.from_dbt("dbt_schema.yml", model="customers")
source_df = gen.generate(rows=500, invalid_ratio=0.08, output_format=ENGINE)

good, bad = proc.run(source_df)
s.assert_reconciliation(source_df, good, bad)
print("\ndbt not_null + accepted_values tests → LakeLogic quality rules. Zero rewrite.")

---
## 2. dlt Adapter — Contract-Driven API Ingestion

**The Problem:** You ingest from GitHub, Stripe, Shopify and 100+ APIs via [dlt](https://dlthub.com). Data arrives with no schema enforcement — bad records flow straight into your warehouse.

**The Solution:** Declare the API directly in your contract's `source.type: dlt` block. LakeLogic extracts the data via dlt's REST API engine, then validates every row through your model and quality rules — all in one `proc.run_source()` call.

In [ ]:
# Install dlt (if not already installed)
import subprocess
import sys

try:
    import dlt
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dlt"])
    import dlt
print(f"dlt v{dlt.__version__} ready")

In [ ]:
# ── The contract declares the API source directly ─────────────
# No separate dlt script needed — the contract IS the config.
#
# GitHub's anonymous API allows only 60 requests/hour per IP (Colab IPs are
# shared, so that budget can already be spent by others). Two things keep this
# demo reliable:
#   1. paginator: single_page  -> ONE request (the first 30 issues), not the
#      ~30 paged calls a full crawl would make
#   2. an optional GITHUB_TOKEN -> raises the limit to 5,000/hour when present
import os

_gh_token = os.environ.get("GITHUB_TOKEN", "").strip()
_creds = f"      token: {_gh_token}" if _gh_token else "      {}"

github_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: github_issues
info:
  title: bronze_github_issues
  domain: engineering
  target_layer: bronze

source:
  type: dlt
  dlt:
    base_url: https://api.github.com
    credentials:
{_creds}
    endpoints:
      - name: issues
        path: repos/dlt-hub/dlt/issues
        paginator: single_page      # one request = first page only (rate-limit friendly)
        params:
          state: open
          per_page: 30

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: repository_url
      type: string
    - name: number
      type: integer
      required: true
    - name: title
      type: string
      required: true
    - name: body
      type: string
    - name: state
      type: string
    - name: url
      type: string
    - name: created_at
      type: string
    - name: updated_at
      type: string

quality:
  row_rules:
    - name: valid_state
      sql: "state IN ('open', 'closed')"
    - name: has_title
      sql: "title IS NOT NULL AND title != ''"

server:
  type: local
  path: "."
  schema_policy:
    evolution: "allow"
    unknown_fields: "drop"
""",
    "06_integrations_demo/github_issues.yaml",
)

print("Contract written with source.type=dlt")
print("  API: https://api.github.com/repos/dlt-hub/dlt/issues (single page, 30 issues)")
print(f"  Auth: {'GITHUB_TOKEN (5,000 req/hr)' if _gh_token else 'anonymous (60 req/hr)'}")
print("  Rules: valid_state, has_title")

In [ ]:
# ── One call: dlt extraction + LakeLogic validation ────────────
# run_source() detects source.type=dlt and:
#   1. Builds a dlt REST API pipeline from the contract config
#   2. Extracts data from the GitHub API
#   3. Loads it into the engine you selected (ENGINE)
#   4. Runs schema validation + quality rules
#   5. Returns good/bad split with reconciliation guarantee

proc = ll.DataProcessor(github_contract, engine=ENGINE)
good = bad = None
try:
    good, bad = proc.run_source()
    counts = proc.last_report.get("counts", {})
    src, g, q = counts.get("source", 0), counts.get("good", 0), counts.get("quarantined", 0)
    print("Contract-driven dlt results:")
    print(f"  Source  : {src} issues from GitHub API")
    print(f"  Good    : {g} (passed all rules)")
    print(f"  Bad     : {q} (quarantined)")
    print(f"  Match   : {src} == {g} + {q} -> {src == g + q}")
    print("\nThe contract IS the config. No dlt script. No manual DataFrame wrangling.")
except Exception as e:
    msg = str(e).lower()
    if "rate limit" in msg or "403" in msg:
        print("GitHub returned HTTP 403 — the anonymous API allows only 60 requests/hour,")
        print("and Colab IPs are shared, so the budget can already be spent by others.")
        print("\nTo raise the limit to 5,000/hour, set a token and re-run the two cells above:")
        print("    import os; os.environ['GITHUB_TOKEN'] = 'ghp_...'  # classic/fine-grained, no scopes needed")
        print("\nThe pattern is unchanged — the contract still IS the config; only the auth header differs.")
    else:
        raise

In [ ]:
print("preview live github data - GOOD DATA")
s.preview(good, 3)

In [ ]:
print("preview live github data - BAD DATA")
s.preview(bad, 3)

---
## 3. Live Streaming Data — Native Connectors

**The Problem:** Data arrives continuously from real-time feeds (wiki edits, clickstreams, market data via WebSocket/Kafka). You need to validate every message against your schema before it lands in the lakehouse.

**The Solution:** LakeLogic ships with native streaming connectors (`SSEConnector`, `WebSocketConnector`, `KafkaConnector`, `WebhookConnector`, and more). Each connector's `.stream()` method yields JSON events which you buffer into a DataFrame and pipe directly through your contract. Below we tap the **public Wikimedia Recent Changes** SSE stream (no auth, no geo-restriction) to capture live edits — the exact same pattern works with any connector.

In [ ]:
# ── Stream live Wikimedia edits using LakeLogic's SSEConnector ───
# Binance geo-blocks cloud IPs (HTTP 451 from Colab), so we tap the public
# Wikimedia Recent Changes stream. Wikimedia rejects requests with no descriptive
# User-Agent (HTTP 403), so we set one; retry=False makes a bad connection fail
# fast instead of looping forever.
# The same pattern works with WebSocketConnector / KafkaConnector / WebhookConnector.
from lakelogic.engines.streaming_connectors import SSEConnector

connector = SSEConnector(
    "https://stream.wikimedia.org/v2/stream/recentchange",
    headers={"User-Agent": "LakeLogic-Colab-Demo/1.0 (https://github.com/LakeLogic/LakeLogic)"},
    retry=False,
)

# Collect 20 live edit events, then close the connection.
# `timestamp` is the event's Unix epoch time (seconds) — the natural watermark
# for a streaming source, exactly like the CDC section further down.
edits = []
for event in connector.stream():
    edits.append(
        {
            "wiki": event.get("wiki"),
            "type": event.get("type"),
            "title": event.get("title"),
            "user": event.get("user"),
            "bot": event.get("bot"),
            "event_time": event.get("timestamp"),
        }
    )
    if len(edits) >= 20:
        break
connector.close()

# Buffer the events into an engine-native frame. `to_frame` builds whatever ENGINE
# the notebook selected at the top (Polars, DuckDB, or Spark) — the notebook body
# never imports a dataframe library directly.
live_df = s.to_frame(edits, engine=ENGINE)
print(f"Captured {s.row_count(live_df)} live Wikimedia edits via SSEConnector")
display(s.preview(live_df, 3))

In [ ]:
# ── Define a contract for real-time edit validation ─────────
# The stream sends one JSON object per Wikimedia edit. The contract enforces the
# shape and business rules every event must satisfy BEFORE it lands downstream:
#   - required fields must be present (wiki, type, title)
#   - `type` must be one of the known recent-change kinds
# Any event that violates a rule is quarantined with the rule that failed;
# everything else flows on. Same reconciliation guarantee as batch.
stream_contract = s.write_contract(
    """
version: 1.0.0
dataset: wiki_edits
info:
  title: bronze_wiki_edits
  domain: platform
  target_layer: bronze

source:
  type: stream
  path: https://stream.wikimedia.org/v2/stream/recentchange

model:
  fields:
    - name: wiki
      type: string
      required: true
    - name: type
      type: string
      required: true
    - name: title
      type: string
      required: true
    - name: user
      type: string
    - name: bot
      type: boolean

quality:
  row_rules:
    - name: known_change_type
      sql: "type IN ('edit', 'new', 'log', 'categorize')"
    - name: has_title
      sql: "title IS NOT NULL AND title != ''"

server:
  type: local
  path: "."
  schema_policy:
    evolution: "allow"
    unknown_fields: "drop"
""",
    "06_integrations_demo/wiki_edits.yaml",
)

print("Contract written for real-time Wikimedia edit validation!")

In [ ]:
# ── Validate live edits through the contract ───────────
proc = ll.DataProcessor(stream_contract, engine=ENGINE)
good, bad = proc.run(live_df)

s.assert_reconciliation(live_df, good, bad)
print("\nLive Wikimedia edits validated in real-time!")
print(f"  Good : {s.row_count(good)} edits passed all rules")
print(f"  Bad  : {s.row_count(bad)} edits quarantined")
display(s.preview(good, 5, columns=["wiki", "type", "title", "user", "bot"]))

---
## Setting up a Live Database for this Demo
To prove LakeLogic natively executes SQL over the wire, we will quickly create a local SQLite database and populate it with tables. (LakeLogic uses exactly the same engine logic for Postgres, MySQL, SQL Server, etc).

In [ ]:
import os

abs_db_path = os.path.abspath("demo.db").replace("\\", "/")
import sqlite3
from datetime import datetime

conn = sqlite3.connect("demo.db")
c = conn.cursor()

# Seed users table (For Section 3)
c.execute("CREATE TABLE IF NOT EXISTS pg_users (id INTEGER, email TEXT, signup_date TEXT)")
c.execute("DELETE FROM pg_users")
c.executemany(
    "INSERT INTO pg_users VALUES (?, ?, ?)",
    [(1, "test@example.com", "2024-01-01"), (2, "invalid_email.com", "2024-01-02")],
)

# Seed cdc_orders table (For Section 5)
c.execute("CREATE TABLE IF NOT EXISTS cdc_orders (id INTEGER, updated_at TEXT)")
c.execute("DELETE FROM cdc_orders")
c.executemany("INSERT INTO cdc_orders VALUES (?, ?)", [(1, "2024-04-10T12:00:00Z"), (2, "2024-04-12T12:00:00Z")])

# Seed massive_orders table (For Section 6)
c.execute("CREATE TABLE IF NOT EXISTS massive_orders (id INTEGER, total REAL)")
c.execute("DELETE FROM massive_orders")
c.executemany("INSERT INTO massive_orders VALUES (?, ?)", [(i, i * 1.5) for i in range(1, 1001)])

# Seed wide_orders_table (For Section 7)
c.execute("CREATE TABLE IF NOT EXISTS wide_orders_table (order_id TEXT, total_amount REAL, huge_json TEXT)")
c.execute("DELETE FROM wide_orders_table")
c.executemany(
    "INSERT INTO wide_orders_table VALUES (?, ?, ?)",
    [("A1", 100.5, '{"data":"blob"}'), ("A2", 50.0, '{"data":"blob"}')],
)

conn.commit()
conn.close()
print("demo.db initialized with seed data!")

---
## 4. Native Database Ingestion (SQLite / Postgres)

**The Problem:** You need to mirror a transactional database (e.g. `users` or `orders` tables) without maintaining brittle JDBC extraction layers or deploying heavy extraction pipelines.

**The Solution:** Declare `source.type: database` and point `source.path` at any SQLAlchemy URI. LakeLogic's native database engine extracts the data at high speed (via ConnectorX / ADBC under the hood) into whichever engine you selected, then validates every row through your model and quality rules — one `proc.run_source()` call, no scaffolding or temporary files.

In [ ]:
# ── Define the schema boundary for Postgres ─────────────
postgres_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: pg_users
info:
  title: bronze_pg_users

source:
  type: database
  path: sqlite:///{abs_db_path}

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: signup_date
      type: string

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%'"
""",
    "06_integrations_demo/postgres_users.yaml",
)

print("Contract written for Postgres extraction")

In [ ]:
# ── Extract directly from DataProcessor ────
# run_source() reads source.type=database, extracts over the wire, and validates
# in the engine you selected (ENGINE) — the same call works for every source type.
proc = ll.DataProcessor("06_integrations_demo/postgres_users.yaml", engine=ENGINE)
res = proc.run_source()

print(f"\nExtracted {res.source_count} rows from the db.")
print(f"Good rows: {res.good_count}")
print(f"Quarantined rows: {res.bad_count}")
display(s.preview(res.good))

---
## 5. Incremental CDC (Change Data Capture)

**The Problem:** Your database has millions of rows. Extracting the full table every minute crushes the database and your pipeline.

**The Solution:** Set `load_mode: incremental` and a `watermark_field`. LakeLogic stores the max watermark safely in the `.lakelogic` state folder and dynamically injects `WHERE updated_at > last_watermark` into the SQL engine before data is even loaded.

In [ ]:
# ── Contract declaring Incremental CDC ─────────────
cdc_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: cdc_orders
info:
  title: bronze_cdc_orders
  target_layer: bronze

source:
  type: database
  load_mode: incremental
  watermark_field: updated_at
  path: sqlite:///{abs_db_path}

# Incremental loads need a run-log backend to persist the watermark between runs.
metadata:
  run_log_dir: "06_integrations_demo/cdc_logs"

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: updated_at
      type: timestamp
      required: true
""",
    "06_integrations_demo/cdc_orders.yaml",
)

print("Contract written with Incremental CDC watermark tracking!")

In [ ]:
# ── Execute Incremental CDC Load ────
proc = ll.DataProcessor("06_integrations_demo/cdc_orders.yaml", engine=ENGINE)
res = proc.run_source()

print(f"\nTotal processed: {res.source_count}")
display(s.preview(res.good, 3))

---
## 6. Massive Initial Loads / Batch Ingestion

**The Problem:** You have a 100GB table in SQL Server or MySQL. Doing a `SELECT *` for the initial load will crash the worker node with an Out-of-Memory (OOM) error before it can evaluate any quality rules.

**The Solution:** Add `options: {fetch_size: N}`. On the **polars** engine, LakeLogic streams the table through a SQLAlchemy iterator — pulling `N` rows at a time, validating each chunk against your contract independently, and combining the results — so peak memory stays flat no matter how large the table is. In production `N` is typically 100,000–500,000; **this demo uses a deliberately small `fetch_size` so the 1,000-row sample table visibly splits into several chunks** (watch for the `Processing database chunk …` log lines).

Memory-bounded ingestion works on every engine, just via different mechanisms:

| Engine | How large tables stay memory-safe | `fetch_size` |
| :--- | :--- | :--- |
| **polars** | driver-side chunk loop (SQLAlchemy iterator) | rows per chunk |
| **spark** | streams partitions across executors; `partition_column` + bounds → parallel read | → JDBC `fetchsize` |
| **duckdb** | native scanner streams the scan vectorised | no-op |

> This section runs on **polars** so the row-chunking is observable. Spark reads via JDBC and needs the dialect's driver jar on the classpath (e.g. `spark.jars.packages='org.postgresql:postgresql'`), so it can't run against this demo's SQLite table without adding one.

In [ ]:
# ── Native Engine Dialect Agnosticism & Batching ─────
# LakeLogic automatically detects dialect extensions via URI (MySQL, SQL Server, SQLite).
batch_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: massive_orders
info:
  title: bronze_massive_orders

source:
  type: database
  path: sqlite:///{abs_db_path}
  options:
    fetch_size: 400   # <- small chunk so this 1,000-row demo table splits into
                      #    visible batches; in production use 100k-500k

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: total
      type: float
""",
    "06_integrations_demo/batch_orders.yaml",
)

print("Contract written with fetch_size configured for batch streams!")

In [ ]:
# ── Execute Batch Ingestion (Iterative) ────
# fetch_size row-chunking is honored by the polars engine, so we run THIS section
# on polars to actually see the streaming iterator split the table into chunks.
# (Every other section respects the notebook's ENGINE selector.)
proc = ll.DataProcessor("06_integrations_demo/batch_orders.yaml", engine="polars")
res = proc.run_source()

good_n, bad_n = s.row_count(res.good), s.row_count(res.bad)
print(f"\nValidated {good_n + bad_n} rows across chunks -> good={good_n}, bad={bad_n}")
print("Each 'Processing database chunk N' line above is one 400-row batch, validated")
print("independently — so peak memory stays flat no matter how big the table is.")
display(s.preview(res.good, 3))

---
## 7. Smart Column Projection (Pushdown)

**The Problem:** The source `orders` table has 150 columns (including heavy JSON blobs), but your analytics pipeline only needs 3 columns. Doing `SELECT *` wastes network bandwidth and memory.

**The Solution:** You do absolutely nothing! LakeLogic's Native Database engines intelligently read your `model.fields` list and dynamically construct precise `SELECT "col1", "col2"` queries. It only extracts exactly what is defined in the contract.

In [ ]:
# ── Automatic Projection Pushdown ──────
# This contract will automatically generate the query:
# SELECT "order_id", "total_amount" FROM "wide_orders_table"
projection_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: wide_orders_table
info:
  title: bronze_narrow_orders

source:
  type: database
  path: sqlite:///{abs_db_path}

model:
  fields:
    # Only these two fields are extracted over the wire!
    - name: order_id
      type: string
    - name: total_amount
      type: float
""",
    "06_integrations_demo/projection_orders.yaml",
)

print("Contract written showcasing smart zero-effort projection pushdown!")

In [ ]:
# ── Execute Smart Projection Extraction ────
proc = ll.DataProcessor("06_integrations_demo/projection_orders.yaml", engine=ENGINE)
res = proc.run_source()

print(f"\nTotal processed: {res.source_count}")
display(s.preview(res.good, 3))

---
## 8. Pre-Phase Column Filtering (Files & APIs)

**The Problem:** Section 7 showed Smart Column Projection for **databases**, where the SQL query is pushed down to the database server. But what about **files** (CSV, Parquet, JSON in S3/ADLS) and **REST APIs** (via dlt)? You can't send a `SELECT` statement to an S3 bucket.

**The Solution:** Use a `phase: pre` transformation. LakeLogic first downloads/reads the full file into an in-memory table, then runs your SQL **before** schema validation or quality rules fire. This gives you the same wildcard/prefix filtering power across *any* source type.

In [ ]:
# ── Pre-Phase Column Filtering ────────────────────────────────
# For files and APIs, use pre-phase SQL to achieve the same
# wildcard/prefix column selection that databases get natively.
#
# Use case: Ingest a wide CSV/Parquet file but only keep columns
# matching a prefix (e.g. 'ice_') or exclude columns with a
# keyword (e.g. '_ex').

pre_filter_contract_yaml = """
version: 1.0.0
dataset: filtered_orders
info:
  title: orders_prefix_filtered

source:
  type: landing
  path: 06_integrations_demo/wide_orders.csv

# This runs AFTER the file is loaded into memory,
# but BEFORE schema validation or quality rules fire.
transformations:
  - phase: pre
    sql: >
      SELECT order_id, status,
             COLUMNS('ice_.*')
      FROM source
      
# For API sources (dlt), the exact same pattern works:
#   source:
#     type: dlt
#     dlt:
#       source: stripe_analytics
#       resource: charges
#   transformations:
#     - phase: pre
#       sql: "SELECT id, amount, COLUMNS('ice_.*') FROM source"

model:
  fields:
    - name: order_id
      type: string
      required: true
    - name: status
      type: string

materialization:
  format: parquet
  target_path: 06_integrations_demo/output/filtered_orders
"""

# Write the contract
s.write_contract(pre_filter_contract_yaml, "06_integrations_demo/pre_filter_orders.yaml")
print("Contract written: 06_integrations_demo/pre_filter_orders.yaml")

# Generate a wide CSV with 'ice_' prefixed columns to demonstrate
import csv
import os

os.makedirs("06_integrations_demo", exist_ok=True)
with open("06_integrations_demo/wide_orders.csv", "w", newline="") as f:
    writer = csv.writer(f)
    # 10 columns: 2 normal + 4 'ice_' prefix + 4 '_ex' suffix
    headers = [
        "order_id",
        "status",
        "ice_region",
        "ice_category",
        "ice_score",
        "ice_flag",
        "amount_ex",
        "tax_ex",
        "notes_ex",
        "internal_ex",
    ]
    writer.writerow(headers)
    writer.writerow(["ORD-001", "shipped", "EU", "electronics", "92", "true", "100", "20", "n/a", "debug"])
    writer.writerow(["ORD-002", "pending", "US", "apparel", "87", "false", "50", "10", "n/a", "test"])
    writer.writerow(["ORD-003", "delivered", "APAC", "food", "95", "true", "200", "40", "n/a", "prod"])

print("\nGenerated wide_orders.csv with 10 columns:")
print("  Normal:  order_id, status")
print("  ice_*:   ice_region, ice_category, ice_score, ice_flag")
print("  *_ex:    amount_ex, tax_ex, notes_ex, internal_ex")
print("\nThe pre-phase SQL will SELECT only order_id, status, and ice_* columns.")
print("The *_ex columns will be automatically excluded before validation.")

### When to Use Which Approach

| Source Type | Method | Where SQL Runs |
| :--- | :--- | :--- |
| **Database** (PostgreSQL, Snowflake, etc.) | `source.query:` | On the **database server** (network-efficient) |
| **Files** (CSV, Parquet, JSON in S3/ADLS) | `transformations: [{phase: pre, sql: ...}]` | In **LakeLogic's engine** (DuckDB/Polars/Spark) |
| **APIs** (via dlt — Stripe, Shopify, etc.) | `transformations: [{phase: pre, sql: ...}]` | In **LakeLogic's engine** (DuckDB/Polars/Spark) |

> **Key Insight:** For databases, push filtering to the server to save bandwidth. For files and APIs, LakeLogic applies the same filtering in-memory after download but before validation.

---
## 8. Cloud Data Sources (Azure, AWS, GCP)

**The Problem:** Your data lives in cloud storage — Azure Data Lake (ADLS), Amazon S3, or Google Cloud Storage. You need to read, validate, and materialize it without writing boilerplate credential code.

**The Solution:** LakeLogic natively resolves `abfss://`, `s3://`, and `gs://` URIs. Its built-in `CloudCredentialResolver` automatically detects credentials from environment variables, service principals, IAM roles, or `az login` — zero manual `storage_options` configuration.

In [ ]:
# ── LakeLogic's built-in cloud credential resolver ──────────
from lakelogic import CloudCredentialResolver

resolver = CloudCredentialResolver()

# Auto-detects from env vars, az login, managed identity, or IAM roles
print("CloudCredentialResolver supports:")
print("  • Azure ADLS/Blob  — abfss://container@account.dfs.core.windows.net/")
print("  • Amazon S3        — s3://bucket/prefix/")
print("  • Google Cloud GCS — gs://bucket/prefix/")
print()
print("Authentication priority (Azure):")
print("  1. Explicit token/key (AZURE_STORAGE_ACCOUNT_KEY, SAS_TOKEN)")
print("  2. Service Principal (AZURE_CLIENT_ID + SECRET + TENANT_ID)")
print("  3. Account key from env var")
print("  4. Azure AD (az login / managed identity / workload identity)")
print()
print("Authentication priority (AWS):")
print("  1. Explicit credentials (AWS_ACCESS_KEY_ID + SECRET)")
print("  2. Environment variables")
print("  3. IAM role (boto3 default credential chain)")
print()
print("Authentication priority (GCP):")
print("  1. GOOGLE_SERVICE_ACCOUNT")
print("  2. GOOGLE_APPLICATION_CREDENTIALS env var")
print("  3. Application Default Credentials (gcloud auth)")

In [ ]:
# ── Azure Data Lake Storage (ADLS Gen2) ─────────────────
# Just set your source.path to an abfss:// URI and LakeLogic handles the rest.
# Credentials are auto-resolved from `az login` or env vars.
#
# The `partition` block is the key optimization:
#   - format: maps to your landing directory structure (strftime tokens)
#   - lookback_days: only scans the last N days instead of the entire lake
#   - This turns a full glob scan into a precise directory lookup

import os

# ── Production: pull from environment variables ─────────────────────
AZURE_STORAGE_ACCOUNT = os.environ.get("AZURE_STORAGE_ACCOUNT", "mystorageaccount")
AZURE_LANDING_CONTAINER = os.environ.get("AZURE_LANDING_CONTAINER", "landing")
AZURE_SILVER_CONTAINER = os.environ.get("AZURE_SILVER_CONTAINER", "silver")

azure_contract_yaml = f"""
version: 1.0.0
dataset: customer_events
info:
  title: bronze_customer_events
  domain: marketing
  target_layer: bronze

source:
  path: abfss://{AZURE_LANDING_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events
  format: parquet
  load_mode: incremental
  watermark_strategy: pipeline_log

  # Partition-aware ingestion — only scan relevant date directories
  partition:
    format: "y_%Y/m_%m/d_%d"        # matches: events/y_2026/m_04/d_16/*.parquet
    lookback_days: 3                  # only scan last 3 days (not the entire lake)
    # start_date: "2026-01-01"        # optional: override for backfills
    # end_date: "2026-01-31"          # optional: override for backfills

model:
  fields:
    - name: event_id
      type: string
      required: true
    - name: customer_id
      type: string
      required: true
    - name: event_type
      type: string
    - name: timestamp
      type: timestamp
      required: true

quality:
  row_rules:
    - name: valid_event
      sql: "event_type IN ('click', 'view', 'purchase', 'signup')"

materialization:
  strategy: append
  format: delta
  partition_by: [event_type]           # Delta table partitioned for fast downstream queries
  target_path: abfss://{AZURE_SILVER_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events

server:
  type: local
  path: "."
"""

print("Azure ADLS contract with partition-aware ingestion:")
print(f"  Storage account: {AZURE_STORAGE_ACCOUNT}")
print(f"  Landing:         abfss://{AZURE_LANDING_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events")
print(f"  Output:          abfss://{AZURE_SILVER_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events")
print()
print("  Landing structure:")
print(f"    abfss://{AZURE_LANDING_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/events/")
print("      y_2026/")
print("        m_04/")
print("          d_14/*.parquet")
print("          d_15/*.parquet")
print("          d_16/*.parquet    <-- only these 3 days scanned (lookback_days: 3)")
print()
print("  Without partition:  scans ALL directories (slow, expensive)")
print("  With partition:     scans only y_2026/m_04/d_14, d_15, d_16 (fast, cheap)")
print()
print("# To execute:")
print("# proc = ll.DataProcessor(contract, engine=ENGINE)")
print("# good, bad = proc.run_source()  # Reads only last 3 days from ADLS")

In [ ]:
# ── Amazon S3 ───────────────────────────────────────
# Same pattern — just swap the URI to s3://

import os

# ── Production: pull from environment variables ─────────────────────
AWS_S3_BUCKET = os.environ.get("AWS_S3_BUCKET", "my-data-lake")
AWS_S3_PREFIX = os.environ.get("AWS_S3_PREFIX", "landing/orders")

s3_contract_yaml = f"""
version: 1.0.0
dataset: order_events
info:
  title: bronze_order_events
  domain: commerce
  target_layer: bronze

source:
  path: s3://{AWS_S3_BUCKET}/{AWS_S3_PREFIX}/*.json
  format: json
  load_mode: incremental
  watermark_strategy: source_mtime

model:
  fields:
    - name: order_id
      type: string
      required: true
    - name: customer_id
      type: string
      required: true
    - name: total_amount
      type: float
    - name: currency
      type: string
    - name: created_at
      type: timestamp
      required: true

quality:
  row_rules:
    - name: positive_amount
      sql: "total_amount > 0"
    - name: valid_currency
      sql: "currency IN ('USD', 'EUR', 'GBP', 'JPY')"

server:
  type: local
  path: "."
"""

print("AWS S3 contract (would run with real credentials):")
print(f"  source: s3://{AWS_S3_BUCKET}/{AWS_S3_PREFIX}/*.json")
print()
print("# To execute:")
print('# contract = s.write_contract(s3_contract_yaml, "commerce/orders.yaml")')
print("# proc = ll.DataProcessor(contract, engine=ENGINE)")
print("# good, bad = proc.run_source()  # Reads from S3 automatically")

In [ ]:
# ── Google Cloud Storage (GCS) ─────────────────────
# Same pattern — just swap to gs://

import os

# ── Production: pull from environment variables ─────────────────────
GCS_BUCKET = os.environ.get("GCS_BUCKET", "my-analytics-bucket")
GCS_PREFIX = os.environ.get("GCS_PREFIX", "sessions")

gcs_contract_yaml = f"""
version: 1.0.0
dataset: user_sessions
info:
  title: bronze_user_sessions
  domain: analytics
  target_layer: bronze

source:
  path: gs://{GCS_BUCKET}/{GCS_PREFIX}/*.parquet
  format: parquet
  load_mode: full

model:
  fields:
    - name: session_id
      type: string
      required: true
    - name: user_id
      type: string
      required: true
    - name: page_views
      type: integer
    - name: duration_seconds
      type: integer
    - name: started_at
      type: timestamp

quality:
  row_rules:
    - name: positive_views
      sql: "page_views >= 0"
    - name: reasonable_duration
      sql: "duration_seconds BETWEEN 0 AND 86400"

server:
  type: local
  path: "."
"""

print("GCP Cloud Storage contract (would run with real credentials):")
print(f"  source: gs://{GCS_BUCKET}/{GCS_PREFIX}/*.parquet")
print()
print("# To execute:")
print('# contract = s.write_contract(gcs_contract_yaml, "analytics/sessions.yaml")')
print("# proc = ll.DataProcessor(contract, engine=ENGINE)")
print("# good, bad = proc.run_source()  # Reads from GCS automatically")

In [ ]:
# ── Databricks Secret Scope Integration ─────────────────
# On Databricks, LakeLogic can pull credentials from secret scopes
# (backed by Azure Key Vault, AWS Secrets Manager, etc.)
from lakelogic.engines.cloud_credentials import DatabricksSecretResolver

print("DatabricksSecretResolver usage (Databricks notebooks only):")
print()
print("# Azure (Key Vault-backed scope):")
print('# resolver = DatabricksSecretResolver.for_cloud("azure", scope="lakelogic")')
print("# options  = resolver.resolve_storage_options(")
print('#     "abfss://silver@myaccount.dfs.core.windows.net/orders/"')
print("# )")
print()
print("# AWS (Secrets Manager-backed scope):")
print('# resolver = DatabricksSecretResolver.for_cloud("aws", scope="lakelogic-aws")')
print('# options  = resolver.resolve_storage_options("s3://my-bucket/silver/")')
print()
print("# GCP (Secret Manager-backed scope):")
print('# resolver = DatabricksSecretResolver.for_cloud("gcp", scope="lakelogic-gcp")')
print('# options  = resolver.resolve_storage_options("gs://my-bucket/silver/")')

## What You Just Did

Eight integrations proving LakeLogic fits into the stack you already have:

- ✅ **dbt adapter** — import `schema.yml`, zero duplicate definitions
- ✅ **dlt adapter** — contract-driven API ingestion with validation built-in
- ✅ **Live streaming** — native SSE / WebSocket / Kafka connectors with real-time validation
- ✅ **Native DB ingestion** — SQLite, Postgres, SQL Server with dialect translation
- ✅ **Incremental CDC** — watermark-driven, no full table scans
- ✅ **Batch ingestion** — chunked extraction for 100GB+ tables
- ✅ **Smart projection pushdown** — only the columns the contract needs
- ✅ **Cloud storage** — Azure / AWS / GCP with one credential resolver

Lines of glue code you would have written: **a frightening number**.

---
## Go Deeper — Explore by Capability

Each notebook below is **self-contained** and maps to one pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities). Pick the one that matters to you most.

| # | Notebook | What You'll See |
|---|---|---|
| 🚀 | **[Quickstart](00_quickstart.ipynb)** | One contract, every row accounted for, PII masked — in 5 minutes |
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

---

**Like what you saw?** ⭐ [Star us on GitHub](https://github.com/LakeLogic/LakeLogic) — it's how we know this matters.